# task_14 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_14/data/")


In [2]:

policies = pd.read_csv(base / "policies.csv")
claims = pd.read_csv(base / "claims.csv")
payments = pd.read_csv(base / "payments.csv")

claims["loss_date"] = pd.to_datetime(claims["loss_date"])
claims["report_date"] = pd.to_datetime(claims["report_date"])
claims["close_date"] = pd.to_datetime(claims["close_date"])

Q2: Among claims where total payments exceed the deductible, compute settlement lag (loss_date → last payment date). Which cause has the shortest median lag?

In [3]:
payments["payment_date"] = pd.to_datetime(payments["payment_date"])

paid_total = payments.groupby("claim_id")["amount_paid"].sum().rename("paid_total")
last_payment = payments.sort_values("payment_date").groupby("claim_id")["payment_date"].last().rename("last_payment_date")

claims_paid = claims.join(paid_total, on="claim_id").join(last_payment, on="claim_id")
claims_paid = claims_paid.merge(policies[["policy_id", "deductible"]], on="policy_id")

# Filter: total_paid > deductible
over_ded = claims_paid[claims_paid["paid_total"] > claims_paid["deductible"]].copy()
over_ded["settle_lag"] = (over_ded["last_payment_date"] - over_ded["loss_date"]).dt.days

q2 = over_ded.groupby("cause")["settle_lag"].median().sort_values().index[0]
print(q2)

Collision


Q5: Which month has the highest total paid losses?

In [4]:
claims_paid["loss_month"] = claims_paid["loss_date"].dt.to_period("M").astype(str)
q5 = str(claims_paid.groupby("loss_month")["paid_total"].sum().sort_values(ascending=False).index[0])
print(q5)


2024-06
